# M2 · ① SIF 아카이브 전처리

`M0_M1_데이터확보/01_...`에서 출처·원본을 확인한 뒤 실행합니다. 받은 XLSX에는 **제조업 등**과 **건설업** 두 사례 시트가 있고, 열 배치가 서로 다릅니다. 두 시트를 공통 열로 정리하되 원본 문장을 임의로 요약하거나 중복 사례를 삭제하지 않습니다.

**산출물:** `data/personal/processed/source_01_sif_cases.parquet`, `data/personal/reports/source_01_preprocessing_summary.json`  
**공유 주의:** ①의 출처표시·변경금지 조건을 확인하기 전까지 가공 결과는 개인 폴더에만 보관합니다.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert ROOT is not None, 'VS Code에서 SANUP-P 폴더를 열고 커널을 .venv로 선택하세요.'
PERSONAL = ROOT / 'data' / 'personal'
RAW = PERSONAL / 'raw' / 'source_01' / 'source_01_sif_archive.xlsx'
PROCESSED = PERSONAL / 'processed' / 'source_01_sif_cases.parquet'
assert RAW.exists(), f'① SIF 원본이 없습니다: {RAW}'
PROCESSED.parent.mkdir(parents=True, exist_ok=True)
print('원본:', RAW.name, f'{RAW.stat().st_size:,} bytes')

## 1. 원본 구조와 헤더 확인

원본 XLSX의 실제 데이터는 제조업 등 시트의 4행, 건설업 시트의 5행부터 시작합니다. `B:J`만 읽고 원본 연번을 보존합니다.

In [ ]:
book = pd.ExcelFile(RAW)
print('시트:', book.sheet_names)
manufacturing = pd.read_excel(RAW, sheet_name='아카이브(제조업등)', header=None, usecols='B:J', skiprows=3)
construction = pd.read_excel(RAW, sheet_name='아카이브(건설업)', header=None, usecols='B:J', skiprows=4)
manufacturing.columns = ['source_row_id', 'industry_major', 'industry_middle', 'industry_minor',
                         'incident_summary', 'causal_object', 'hazard_work', 'trigger_factor',
                         'risk_reduction_measures']
construction.columns = ['source_row_id', 'construction_phase', 'work_name', 'unit_work',
                        'accident_type', 'incident_summary', 'causal_object', 'trigger_factor',
                        'risk_reduction_measures']
print('제조업 등:', manufacturing.shape, '건설업:', construction.shape)
assert manufacturing['source_row_id'].notna().all() and construction['source_row_id'].notna().all()

## 2. 열 이름 통일과 전처리

- 원본의 연번과 시트 구분으로 안정적인 `record_id`를 만듭니다.
- 문자열의 앞뒤 공백과 눈에 보이지 않는 BOM·제로폭 문자만 제거합니다. 원본의 `▶`·줄바꿈·문장 내용은 유지합니다.
- 건설업의 공종·작업명·단위작업명과 제조업 등의 업종 대·중·소분류는 별도 열로 남깁니다. 없는 값은 결측으로 둡니다.
- 숫자형 연속 변수는 없으므로 IQR로 사고를 제거하지 않습니다. 긴 설명문은 글자 수만 점검합니다.

In [ ]:
from pandas import NA

m = manufacturing.copy()
m['sheet_group'] = '제조업 등'
m['accident_type'] = NA
m['construction_phase'] = NA
m['work_name'] = NA
m['unit_work'] = NA
m['record_id'] = m['source_row_id'].astype(int).map(lambda n: f'SIF-M-{n:04d}')

c = construction.copy()
c['sheet_group'] = '건설업'
c['industry_major'] = '건설업'
c['industry_middle'] = NA
c['industry_minor'] = NA
c['hazard_work'] = NA
c['record_id'] = c['source_row_id'].astype(int).map(lambda n: f'SIF-C-{n:04d}')

columns = ['record_id', 'sheet_group', 'source_row_id', 'industry_major', 'industry_middle',
           'industry_minor', 'construction_phase', 'work_name', 'unit_work', 'accident_type',
           'incident_summary', 'causal_object', 'hazard_work', 'trigger_factor',
           'risk_reduction_measures']
cases = pd.concat([m[columns], c[columns]], ignore_index=True)
for column in cases.select_dtypes(include=['object', 'string']).columns:
    cases[column] = cases[column].astype('string').str.replace('﻿', '', regex=False).str.replace('​', '', regex=False).str.strip()
    cases[column] = cases[column].replace('', NA)
# '원인 미상' 단독 표기는 구체적 예방대책이 아니라 원본의 미제공 표시다.
measure_label=cases['risk_reduction_measures'].fillna('').str.replace(r'\s+','',regex=True)
cases['risk_measures_available']=~measure_label.str.fullmatch(r'▶?원인미상')
assert int((~cases['risk_measures_available']).sum())==13
assert cases['record_id'].is_unique
assert cases['incident_summary'].notna().all()
print('전체 사례:', len(cases))
print(cases.groupby('sheet_group').size().rename('사례 수'))

## 3. 결측·중복·텍스트 길이 점검

건설업에 업종 중분류가 없고 제조업 등에는 공종이 없는 것은 원본 구조상 **해당 없음**입니다. 이 결측을 임의의 업종이나 공종으로 채우지 않습니다. 같은 사고 개요가 반복되어도 다른 원본 연번이면 자동 삭제하지 않습니다.

In [ ]:
profile = cases.isna().sum().to_frame('결측 수')
profile['결측률'] = (profile['결측 수'] / len(cases)).round(4)
display(profile)
duplicate_text = int(cases['incident_summary'].duplicated(keep=False).sum())
print('중복 record_id:', int(cases['record_id'].duplicated().sum()))
print('같은 재해개요 문장이 있는 행:', duplicate_text, '(자동 삭제하지 않음)')
lengths = cases[['incident_summary', 'trigger_factor', 'risk_reduction_measures']].apply(lambda s: s.str.len())
display(lengths.describe().round(1))

## 4. 개인 정제본과 전처리 기록 저장

저장은 원본을 덮어쓰지 않습니다. 전후 행 수와 결정 이유를 함께 기록합니다.

In [ ]:
import json

cases.to_parquet(PROCESSED, index=False)
summary = {
    'source_id': '01',
    'raw_file': str(RAW.relative_to(ROOT)),
    'processed_file': str(PROCESSED.relative_to(ROOT)),
    'raw_rows': {'manufacturing_etc': len(manufacturing), 'construction': len(construction)},
    'processed_rows': len(cases),
    'missing_by_column': {name: int(value) for name, value in cases.isna().sum().items()},
    'duplicate_record_ids': int(cases['record_id'].duplicated().sum()),
    'same_summary_rows': duplicate_text,
    'unclassified_causal_object_rows': int(cases['causal_object'].eq('분류 불가').sum()),
    'risk_measures_unavailable_rows': int((~cases['risk_measures_available']).sum()),
    'rules': ['시트별 원본 열 매핑', '앞뒤 공백·BOM·제로폭 문자 제거', '구조상 해당 없는 필드는 결측 유지',
              '중복 문장·긴 텍스트·이상치 자동 삭제 없음'],
}
report = PERSONAL / 'reports' / 'source_01_preprocessing_summary.json'
report.parent.mkdir(parents=True, exist_ok=True)
report.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('정제본:', PROCESSED, '전처리 기록:', report)

## 전처리 결과와 판단 근거

### 이번 단계에서 한 일

원본 XLSX의 제조업 등·건설업 시트를 각각 읽고 열 배치를 확인한 뒤, 공통 열 이름과 record_id를 만들었다. 원본은 덮어쓰지 않고 정제본을 data/personal/processed/source_01_sif_cases.parquet에 따로 저장했다.

- 제조업 등 2,573건, 건설업 3,459건, 합계 6,032건을 유지했다.
- 공백·BOM·제로폭 문자만 정리했다. 문장·기호·원인·감소대책의 의미를 바꾸거나 짧게 줄이지 않았다.
- 구조상 해당하지 않는 열은 결측으로 두고, 중복 사례문장과 긴 텍스트를 자동 삭제하지 않았다.
- 감소대책이 원인 미상뿐인 13건은 원문 그대로 두고 risk_measures_available=False로 표시했다.

### 확인한 값과 알게 된 점

- record_id 중복은 0건이고, 재해개요 문장 전체 기준 중복은 2행이다. 서로 다른 원본 행일 수 있어 중복 문장만으로 사례를 합치지 않았다.
- 건설업에는 제조업 업종 세분류 대신 공정·작업명이 있고, 제조업 등에는 건설 공정명이 없다. 각 시트 구조가 달라 생긴 결측이므로 결측률을 데이터 오류로 해석하지 않는다.
- 제조업 등과 건설업을 합한 사례 파일은 수집된 사례 아카이브다. 업종별 발생 위험률·전체 재해 발생률을 뜻하지 않는다.
- 기인물 분류 불가가 398건(전체의 약 6.6%)이다. 상위 기인물 그래프나 검색 결과에서 이 범주를 숨기지 않는다.
- 글자 수는 재해개요 평균 99.7자·최대 486자, 재해유발요인 평균 60.4자·최대 197자, 감소대책 평균 106.7자·최대 382자였다. 이는 요약 품질이 아니라 텍스트 길이 점검이다.

### 다음 단계에서 사용할 것

분석에는 업종·시트 구분·기인물·고위험작업·재해유발요인을 사용하고, RAG에서는 record_id와 원문 필드를 함께 보존한다. 감소대책이 비어 있거나 원인 미상으로 표시된 경우 구체적인 예방조치 근거로 출력하지 않는다.

### 유의할 점

① SIF는 공공누리 제3유형(출처표시·변경금지)으로 안내된다. 원본 XLSX는 그대로 유지한다. 이 노트북의 정제본은 개인 작업 폴더에만 두고, 청크·요약·임베딩·LLM 입력·앱 노출을 외부에 공유하거나 재배포할 수 있는지는 별도 이용조건 확인 전까지 단정하지 않는다. 문서가 금지하는 변경을 전처리했으니 허용된다고 해석하지 않는다.

### 재현 파일

- 정제본: data/personal/processed/source_01_sif_cases.parquet
- 전처리 건수·결측 요약: data/personal/reports/source_01_preprocessing_summary.json
